# TabICLv2 (Inria) — Regression

Demonstrates **TabICL**, the zero-shot tabular foundation model, on the shared
retail/CPG regression datasets. TabICLv2 is **self-hosted**: model weights are
downloaded from Hugging Face and inference runs locally in a single forward pass.

> **License note:** TabICLv2 is released under a [commercially permissive license](https://huggingface.co/datasets/choosealicense/licenses/blob/main/markdown/bsd-3-clause.md) 

**Compute:** GPU cluster recommended (weights run on GPU). If GPU serverless used, a single 1xA10 is enough. CPU works but is slow.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.


In [0]:
%pip install tabicl==2.1.1 --quiet

In [0]:
dbutils.library.restartPython()

## Configuration


In [0]:
import os, sys

CATALOG = "tabular_fm"
SCHEMA = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Import libraries

TabICLv2 downloads the weights from HuggingFace directly when loading train data into context using an sklearn-style estimator (i.e. .fit)


In [0]:
import numpy as np
import pandas as pd
import mlflow

from sklearn.preprocessing import OrdinalEncoder

import torch
from tabicl import TabICLRegressor

from evaluation import (
    split_xy, regression_metrics, train_baselines_regression,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


In [0]:
def evaluate_regression(table_name, target, task_name, test_size=0.2):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(df, target=target, test_size=test_size)

    # Encode categorical columns (TabICL requires numeric inputs)
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train = X_train.copy()
        X_test = X_test.copy()
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
        X_test[cat_cols] = enc.transform(X_test[cat_cols])

    n_train, n_test, n_features = len(X_train), len(X_test), X_train.shape[1]

    with mlflow.start_run(run_name=f"{task_name}_tabicl"):
        mlflow.log_params({
            "vendor": "tabicl", "model_type": "TabICLRegressor",
            "task": task_name, "problem_type": "regression",
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        device = "cuda" if torch.cuda.is_available() else "cpu"
        reg = TabICLRegressor(device=device)
        reg.fit(X_train, y_train)
        y_pred = reg.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="tabicl", task=task_name, problem_type="regression",
                   model_name="TabICLRegressor", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] TabICL: rmse={metrics['rmse']:.4f} "
          f"mae={metrics['mae']:.4f} r2={metrics['r2']:.4f}")

    for name, m in train_baselines_regression(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type="regression",
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: rmse={m['rmse']:.4f} r2={m['r2']:.4f}")
    return metrics

## Price Elasticity


In [0]:
_ = evaluate_regression(
    table_name="price_elasticity_train",
    target="price_elasticity",
    task_name="price_elasticity",
)

## Supplier Lead Time


In [0]:
_ = evaluate_regression(
    table_name="supplier_lead_time_train",
    target="actual_lead_time_days",
    task_name="supplier_lead_time",
)

## Results


In [0]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type = 'regression'")
         .orderBy("task", "vendor", "model_name")
)

In [0]:
RESULTS_TABLE